## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


## Para correr con extensión de Colab en Visual Studio Code

In [ ]:
import plotly.io as pio

# Set the renderer explicitly for VS Code Notebooks
pio.renderers.default = "vscode"  # alternative: 'notebook_mimetype'


## 0.2 Init kaggle


In [ ]:
!pip install uv
!uv pip install -q kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 49.4 MB/s eta 0:00:00:00:0100:01


In [ ]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


# 1 Modelos Naif  Polars
Polars es la moderna y eficiente libreria que reemplaza a Pandas

## 1.1 Inicio  Polars

In [ ]:
import os as os
import polars as pl

In [ ]:
# defino los parametros
PARAM = {'experimento':'naif01',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [ ]:
# creo la carpeta del experimento
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/naif01


In [ ]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/datasets/sell-in.txt.gz', separator="\t")
display(dataset)

periodo,customer_id,product_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn
i64,i64,i64,i64,i64,f64,f64
201701,10234,20524,0,2,0.053,0.053
201701,10032,20524,0,1,0.13628,0.13628
201701,10217,20524,0,1,0.03028,0.03028
201701,10125,20524,0,1,0.02271,0.02271
201701,10012,20524,0,11,1.54452,1.54452
…,…,…,…,…,…,…
201912,10105,20853,0,1,0.0223,0.0223
201912,10092,20853,0,1,0.00669,0.00669
201912,10006,20853,0,7,0.02898,0.02898


In [ ]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21295,201701,0.00699
21296,201708,0.00651
21297,201701,0.00579


In [ ]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/datasets/product_id_apredecir201912.txt', separator="\t")
display(tb_apredecir)

product_id
i64
20001
20002
20003
20004
20005
…
21263
21265
21266


In [ ]:
# filtrado
print(tb_ventas.height)

tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)

print(tb_ventas.height)

31243
22349


## 1.2 En febrero se venderá lo mismo que en diciembre

En 202002  se venderá lo mismo que en 201912
<br>  tn( t+2) = tn(t)

In [ ]:
ultimo_periodo = 201912

tb_ultimo = tb_ventas.filter(pl.col("periodo") == ultimo_periodo).select(["product_id", "tn"])
display(tb_ultimo)

product_id,tn
i64,f64
20001,1504.68856
20002,1087.30855
20003,892.50129
20004,637.90002
20005,593.24443
…,…
21263,0.0127
21265,0.05007
21266,0.05121


In [ ]:
archivo = "naif_ultimo_polars.csv"
tb_ultimo.write_csv(archivo)

In [ ]:
# submit a Kaggle
kaggle_submit(PARAM['kaggle_competition'], archivo, "naif = 201912, polars"  )

## 1.3 En febrero se vendera el promedio del año 2019
En 202002 se venderá lo mismo que el promedio de [201901, 201912]
<br>tn( t+2) = mean(  tn[ t-11, t ] )

In [ ]:
primer_periodo = 201901
ultimo_periodo = 201912

tb_meses12 = tb_ventas.filter( pl.col("periodo").is_between(primer_periodo,ultimo_periodo)).group_by("product_id").agg(
 pl.col("tn").mean().alias("tn"))

tb_meses12 = tb_meses12.select(["product_id", "tn"])
display(tb_meses12)


product_id,tn
i64,f64
20001,1454.73272
20002,1175.437142
20003,784.976408
20004,627.215328
20005,668.270104
…,…
21263,0.029993
21265,0.089541
21266,0.094659


In [ ]:
archivo = "naif_meses12_polars.csv"
tb_meses12.write_csv(archivo)

In [ ]:
# submit a Kaggle
kaggle_submit(PARAM['kaggle_competition'], archivo, "naif promedio ultimos 12 meses, polars"  )

## 1.4 En febrero se vendera lo mismo que el febrero anterior
En 202002 se venderá lo mismo que en 201902
<br>tn( t+2) = tn(t-10)

En realidad, como el producto podría ser nuevo y no haber existido en 201902
<br> si no llegara a existir, le calculo el promedio del año

In [ ]:
primer_periodo = 201901
ultimo_periodo = 201912
tb_meses12 = tb_ventas.filter( pl.col("periodo").is_between(primer_periodo,ultimo_periodo)).group_by("product_id").agg(
 pl.col("tn").mean().alias("tn"))

tb_meses12 = tb_meses12.select(["product_id", "tn"])

ultimo_periodo = 201902
tb_febrero = tb_ventas.filter(pl.col("periodo") == ultimo_periodo).select(["product_id", "tn"])


# Update table1 using table2
tb_201902 = (
    tb_meses12
    .join(tb_febrero, on="product_id", how="left", suffix="_update")
    .with_columns(
        pl.coalesce([pl.col("tn_update"), pl.col("tn")]).alias("tn")
    )
    .drop("tn_update")
)

display( tb_201902)


product_id,tn
i64,f64
20001,1259.09363
20002,1043.01349
20003,758.32657
20004,441.70332
20005,409.8995
…,…
21263,0.05927
21265,0.089541
21266,0.094659


In [ ]:
archivo= "naif_201902_polars.csv"
tb_201902.write_csv(archivo)

In [ ]:
# submit a Kaggle
kaggle_submit(PARAM['kaggle_competition'], archivo, "naif mismo mes año anterior, polars" )

# 2 Modelos Naif  DuckDB


DuckDB permite trabajar con lenguaje SQL  en tablas que crea en la memoria RAM

## 2.1 Inicio  DuckDB

In [ ]:
import os as os
import duckdb

In [ ]:
# defino los parametros
PARAM = {'experimento':'naif02',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [ ]:
# creo la carpeta del experimento
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/naif02


In [ ]:
con = duckdb.connect()

In [ ]:
# creo las ventas por producto

con.execute("""
    CREATE TABLE tb_ventas AS
    SELECT
        product_id,
        periodo,
        SUM(tn) AS tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    GROUP BY product_id, periodo
    ORDER BY product_id, periodo
""")

# primeros 13 meses
print(con.sql("SELECT * FROM tb_ventas LIMIT 13").df())


    product_id  periodo          tn
0        20001   201701   934.77222
1        20001   201702   798.01620
2        20001   201703  1303.35771
3        20001   201704  1069.96130
4        20001   201705  1502.20132
5        20001   201706  1520.06539
6        20001   201707  1030.67391
7        20001   201708  1267.39462
8        20001   201709  1316.94604
9        20001   201710  1439.75563
10       20001   201711  1580.47401
11       20001   201712  1049.38860
12       20001   201801  1169.07532


In [ ]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002

con.execute("""
    CREATE TABLE tb_apredecir AS
    SELECT *
    FROM read_csv_auto('/content/datasets/product_id_apredecir201912.txt')
""")



tb_apredecir = pl.read_csv('/content/datasets/product_id_apredecir201912.txt', separator="\t")
display(tb_apredecir)

product_id
i64
20001
20002
20003
20004
20005
…
21263
21265
21266


In [ ]:
# filtrado

print(con.sql("SELECT count(*) FROM tb_ventas").df())

con.execute("""
    CREATE OR REPLACE TABLE tb_ventas AS
    SELECT  v.*
    FROM tb_ventas v,
         tb_apredecir ap
    WHERE  v.product_id = ap.product_id
""")

print(con.sql("SELECT count(*) FROM tb_ventas").df())


   count_star()
0         31243
   count_star()
0         22349


## 2.2 En febrero se venderá lo mismo que en diciembre

En 202002  se venderá lo mismo que en 201912
<br>  tn( t+2) = tn(t)

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_ultimo AS
    SELECT product_id,
           tn
    FROM   tb_ventas
    WHERE  periodo= 201912
""")

In [ ]:

archivo = 'naif_ultimo_duckdb.csv'
con.execute("""
COPY tb_ultimo TO  '{archivo}' (HEADER, DELIMITER ',');
""")

In [ ]:
# submit a Kaggle
kaggle_submit(PARAM['kaggle_competition'], archivo, "naif = 201912, DuckDB" )


## 2.3 En febrero se venderá el promedio del año 2019
En 202002 se venderá lo mismo que el promedio de [201901, 201912]
<br>tn( t+2) = mean(  tn[ t-11, t ] )

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_meses12 AS
    SELECT product_id,
           AVG( tn ) as tn
    FROM   tb_ventas
    WHERE  periodo between 201901 AND 201912
    GROUP BY product_id
""")

In [ ]:
archivo = 'naif_meses12_duckdb.csv'

con.execute("""
COPY tb_meses12 TO '{archivo}' (HEADER, DELIMITER ',');
""")

In [ ]:
# submit a Kaggle
kaggle_submit(PARAM['kaggle_competition'], archivo, "naif promedio ultimos 12 meses, DuckDB" )


## 2.4 En febrero se venderá lo mismo que el febrero anterior
En 202002 se venderá lo mismo que en 201902
<br>tn( t+2) = tn(t-10)

En realidad, como el producto podría ser nuevo y no haber existido en 201902
<br> si no llegara a existir, le calculo el promedio del año

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_201902 AS
    SELECT prom.product_id,
           COALESCE( feb.tn, prom.tn)
    FROM   tb_meses12  prom
    LEFT JOIN (SELECT * FROM tb_ventas WHERE periodo= 201902) feb
      ON feb.product_id = prom.product_id
""")

In [ ]:
archivo = 'naif_201902_duckdb.csv'

con.execute("""
COPY tb_201902 TO '{archivo}' (HEADER, DELIMITER ',');
""")

In [ ]:
# submit a Kaggle
kaggle_submit(PARAM['kaggle_competition'], archivo, "naif mismo mes año anterior, DuckDB" )


# 3 Modelos Naif  Rlang


Pasar a lenguaje R

## 3.1 Inicio  Rlang

In [1]:
require("data.table")

if(! require("R.utils") ) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘R.utils’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach,

In [2]:
# defino funcion para hacer submits a Kaggle

kaggle_submit <- function(competencia, archivo, mensaje) {

  msg <- paste0("'", mensaje, "'")
  comando <- paste( "kaggle competitions submit", "-c", competencia, "-f", archivo, "-m", msg)

  system(comando, intern=TRUE)
}

In [3]:
# defino los parametros
PARAM <- list(
  experimento= "naif03",
  kaggle_competition = "labo-iii-2026-ba"
)

In [4]:
# creo la carpeta del experimento
ruta <- paste0("/content/buckets/b1/", PARAM$experimento)
dir.create(ruta, showWarnings= FALSE)
setwd( ruta )

In [5]:
# creo las ventas por producto
tb_sellin <- fread("/content/datasets/sell-in.txt.gz" )

tb_ventas <- tb_sellin[, list("tn"= sum(tn)), list(product_id, periodo)]


In [6]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002

tb_apredecir <- fread("/content/datasets/product_id_apredecir201912.txt")

In [7]:
# filtrado

nrow(tb_ventas)
tb_ventas <- tb_ventas[ product_id %in%  tb_apredecir$product_id]
nrow(tb_ventas)


[1] 31243

[1] 22349

## 3.2 En febrero se venderá lo mismo que en diciembre

En 202002  se venderá lo mismo que en 201912
<br>  tn( t+2) = tn(t)

In [8]:
tb_ultimo <- tb_ventas[ periodo==201912, list(product_id, tn)]

In [9]:

archivo <- "naif_ultimo_rlang.csv"
fwrite( tb_ultimo, archivo)

In [10]:
# submit a Kaggle
kaggle_submit(PARAM$kaggle_competition, archivo, "naif = 201912, Rlang" )


[1] "Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)"
[2] "Successfully submitted to Labo III, 2026 BA"

## 3.3 En febrero se venderá el promedio del año 2019
En 202002 se venderá lo mismo que el promedio de [201901, 201912]
<br>tn( t+2) = mean(  tn[ t-11, t ] )

In [11]:
tb_meses12 <- tb_ventas[ periodo>= 201901 & periodo<=201912,
  list("tn"= mean(tn)),
  product_id
]

In [12]:
archivo <- "naif_meses12_rlang.csv"
fwrite( tb_meses12, archivo )

In [13]:
# submit a Kaggle
kaggle_submit(PARAM$kaggle_competition, archivo, "naif promedio ultimos 12 meses, Rlang" )



[1] "Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)"
[2] "Successfully submitted to Labo III, 2026 BA"

## 3.4 En febrero se venderá lo mismo que el febrero anterior
En 202002 se venderá lo mismo que en 201902
<br>tn( t+2) = tn(t-10)

En realidad, como el producto podría ser nuevo y no haber existido en 201902
<br> si no llegara a existir, le calculo el promedio del año

In [14]:
tb_201902 <- tb_meses12[  tb_ventas[ periodo==201902,],
  on="product_id",
  tn:= fcoalesce(i.tn, tn)
]


In [15]:
archivo <- "naif_201902_rlang.csv"
fwrite( tb_201902, archivo)

In [16]:
# submit a Kaggle
kaggle_submit(PARAM$kaggle_competition, archivo, "naif mismo mes año anterior, Rlang" )

[1] "Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)"
[2] "Successfully submitted to Labo III, 2026 BA"